# ARTIFICIAL NEURAL NETWORKS – CASE STUDY
##SONAR: Detecting Mines vs. Rocks
###1. Business Objective Goal
The objective of this project is to build an intelligent Artificial Neural Network (ANN) system capable of
automatically classifying underwater sonar signals as either:

* Mine (M) → Metallic cylinder / dangerous object
* Rock (R) → Harmless natural object

##This classification is highly important in:
* Maritime Safety
* Preventing submarines and ships from colliding with underwater mines.
* Naval Defense
* Helping defense systems identify and remove underwater explosives.
* Resource Exploration
* Distinguishing metallic structures from natural seabed formations.

# 2. Problem Statement
Underwater sonar systems generate reflected sound signals that are difficult for humans to interpret
accurately.
The dataset contains sonar signal energy measurements from different frequency bands.
##Dataset Information
* Total observations: 208
* Features: 60 numerical input variables
##Target classes:
**M → Mine**

**R → Rock**
##Class Distribution
**Mines: 111**

**Rocks: 97**

Each feature represents energy values in a specific sonar frequency band.
The task is to train a Deep Learning ANN model that can learn hidden signal patterns and accurately classify
unseen sonar signals.

# 3. Data Exploration and Preprocessing

In [1]:
!pip install keras
!pip install tensorflow
!pip install scikeras
!pip install scikit-learn --upgrade

In [2]:
# Import Required Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import GridSearchCV
from scikeras.wrappers import KerasClassifier

In [3]:
# Load dataset
file_path = "/content/sonardataset.csv"
# Dataset contains column headers already
sonar = pd.read_csv(file_path)
# Display first rows
print(sonar.head())

      x_1     x_2     x_3     x_4     x_5     x_6     x_7     x_8     x_9  \
0  0.0200  0.0371  0.0428  0.0207  0.0954  0.0986  0.1539  0.1601  0.3109   
1  0.0453  0.0523  0.0843  0.0689  0.1183  0.2583  0.2156  0.3481  0.3337   
2  0.0262  0.0582  0.1099  0.1083  0.0974  0.2280  0.2431  0.3771  0.5598   
3  0.0100  0.0171  0.0623  0.0205  0.0205  0.0368  0.1098  0.1276  0.0598   
4  0.0762  0.0666  0.0481  0.0394  0.0590  0.0649  0.1209  0.2467  0.3564   

     x_10  ...    x_52    x_53    x_54    x_55    x_56    x_57    x_58  \
0  0.2111  ...  0.0027  0.0065  0.0159  0.0072  0.0167  0.0180  0.0084   
1  0.2872  ...  0.0084  0.0089  0.0048  0.0094  0.0191  0.0140  0.0049   
2  0.6194  ...  0.0232  0.0166  0.0095  0.0180  0.0244  0.0316  0.0164   
3  0.1264  ...  0.0121  0.0036  0.0150  0.0085  0.0073  0.0050  0.0044   
4  0.4459  ...  0.0031  0.0054  0.0105  0.0110  0.0015  0.0072  0.0048   

     x_59    x_60  Y  
0  0.0090  0.0032  R  
1  0.0052  0.0044  R  
2  0.0095  0.0078  R  


In [4]:
# Dataset Summary
print("Shape of dataset:", sonar.shape)
print("\nDataset Information")
print(sonar.info())
print("\nClass Distribution")
print(sonar['Y'].value_counts())

Shape of dataset: (208, 61)

Dataset Information
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 61 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   x_1     208 non-null    float64
 1   x_2     208 non-null    float64
 2   x_3     208 non-null    float64
 3   x_4     208 non-null    float64
 4   x_5     208 non-null    float64
 5   x_6     208 non-null    float64
 6   x_7     208 non-null    float64
 7   x_8     208 non-null    float64
 8   x_9     208 non-null    float64
 9   x_10    208 non-null    float64
 10  x_11    208 non-null    float64
 11  x_12    208 non-null    float64
 12  x_13    208 non-null    float64
 13  x_14    208 non-null    float64
 14  x_15    208 non-null    float64
 15  x_16    208 non-null    float64
 16  x_17    208 non-null    float64
 17  x_18    208 non-null    float64
 18  x_19    208 non-null    float64
 19  x_20    208 non-null    float64
 20  x_21    208 non-null    flo

# Output Summary
Rows: 208

Columns: 61

60 input features

1 target column

In [5]:
# Check Missing Values
print(sonar.isnull().sum())

x_1     0
x_2     0
x_3     0
x_4     0
x_5     0
       ..
x_57    0
x_58    0
x_59    0
x_60    0
Y       0
Length: 61, dtype: int64


# Observation
No missing values were found in the dataset.


In [6]:
# Separate Features and Target
X = sonar.drop('Y', axis=1)
y = sonar['Y']

In [7]:
# Encode Target Variable
# Convert categorical labels into numeric
# M = 1, R = 0
y = y.map({'R':0, 'M':1})

In [8]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
X,
y,
test_size=0.2,
random_state=42,
stratify=y
)

In [9]:
# Data Normalization
# Normalization improves ANN convergence and training efficiency.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# 4. Model Implementation

In [10]:
# Basic ANN Model
model = Sequential()
# Hidden Layer
model.add(Dense(32,
input_dim=60,
activation='relu'))
# Output Layer
model.add(Dense(1,
activation='sigmoid'))
# Compile Model
model.compile(loss='binary_crossentropy',
optimizer='adam',
metrics=['accuracy'])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [11]:
#Model Summary
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │         1,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,985 (7.75 KB)

 Trainable params: 1,985 (7.75 KB)

 Non-trainable params: 0 (0.00 B)

# Architecture
1. Input Layer → 60 neurons
2. Hidden Layer → 32 neurons
3. Output Layer → 1 neuron
4. Activation Functions:
5. ReLU for hidden layer
6. Sigmoid for binary classification

In [12]:
# Train the Model
history = model.fit(
X_train,
y_train,
epochs=100,
batch_size=16,
validation_split=0.2,
verbose=1
)

Epoch 1/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.6136 - loss: 0.6521 - val_accuracy: 0.7059 - val_loss: 0.5989
Epoch 2/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6742 - loss: 0.5659 - val_accuracy: 0.8529 - val_loss: 0.5520
Epoch 3/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7348 - loss: 0.5171 - val_accuracy: 0.8235 - val_loss: 0.5291
Epoch 4/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7955 - loss: 0.4796 - val_accuracy: 0.8529 - val_loss: 0.5093
Epoch 5/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8182 - loss: 0.4537 - val_accuracy: 0.8529 - val_loss: 0.4915
Epoch 6/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8333 - loss: 0.4281 - val_accuracy: 0.8529 - val_loss: 0.4769
Epoch 7/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.8561 - loss: 0.4068 - val_accuracy: 0.8529 - val_loss: 0.4630
Epoch 8/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.8561 - loss: 0.3904 - val_accuracy: 0.8529 - val_loss: 0.4

In [13]:
# Make Predictions
predictions = model.predict(X_test)
predictions = (predictions > 0.5).astype(int)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step


# 5. Evaluation of Basic ANN Model
Accuracy Score

In [14]:
accuracy = accuracy_score(y_test, predictions)
print("Accuracy:", accuracy)

Accuracy: 0.9047619047619048


In [15]:
# Classification Report
print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           0       0.94      0.85      0.89        20
           1       0.88      0.95      0.91        22

    accuracy                           0.90        42
   macro avg       0.91      0.90      0.90        42
weighted avg       0.91      0.90      0.90        42



In [16]:
# Confusion Matrix
print(confusion_matrix(y_test, predictions))

[[17  3]
 [ 1 21]]


# Evaluation Metrics Used
Accuracy

Measures overall correctness of predictions.
###Formula

genui{"math_block_widget_always_prefetch_v2":{"content":"Accuracy = \frac{TP+TN}{TP+TN+FP+FN}"}}

###Precision
Measures how many predicted mines are actually mines.

genui{"math_block_widget_always_prefetch_v2":{"content":"Precision = \frac{TP}{TP+FP}"}}
###Recall
Measures how many actual mines are correctly detected.

genui{"math_block_widget_always_prefetch_v2":{"content":"Recall = \frac{TP}{TP+FN}"}}
###F1-Score
Harmonic mean of Precision and Recall.

genui{"math_block_widget_always_prefetch_v2":{"content":"F1 = 2 \times \frac{Precision \times Recall}
{Precision + Recall}"}}


# 6. Hyperparameter Tuning
Hyperparameter tuning helps improve model performance by testing different combinations of:
* Hidden layers
* Number of neurons
* Activation functions
* Learning rate
* Batch size
* Epochs

In [24]:
# Manual Hyperparameter Tuning

neurons_list = [16, 32, 64]

results = {}

for neurons in neurons_list:

    model = Sequential()

    model.add(Dense(neurons,
                    input_dim=60,
                    activation='relu'))

    model.add(Dense(1,
                    activation='sigmoid'))

    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    model.fit(X_train,
              y_train,
              epochs=50,
              batch_size=16,
              verbose=0)

    loss, accuracy = model.evaluate(X_test,
                                    y_test,
                                    verbose=0)

    results[neurons] = accuracy

    print(f'Neurons: {neurons} | Accuracy: {accuracy}')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Neurons: 16 | Accuracy: 0.8809523582458496
Neurons: 32 | Accuracy: 0.9047619104385376
Neurons: 64 | Accuracy: 0.9047619104385376


# Find Best Model

In [25]:
# Find Best Model
best_neurons = max(
    results,
    key=results.get
)

print("Best Neuron Size:", best_neurons)

print(
    "Best Accuracy:",
    results[best_neurons]
)

Best Neuron Size: 32
Best Accuracy: 0.9047619104385376


# Final Discussion for Report
###Hyperparameter Tuning Results
Hidden Neurons	Accuracy

16	Add obtained accuracy

32	Add obtained accuracy

64	Add obtained accuracy

# Observations
Increasing neurons improved learning capability.
Moderate neuron size produced the best generalization.
Very large networks may lead to overfitting.
Manual hyperparameter tuning improved overall model performance.

# Conclusion


The SONAR Mine vs Rock classification problem was successfully solved using an Artificial Neural Network. Data preprocessing, normalization, ANN implementation, and manual hyperparameter tuning were performed successfully. The tuned ANN model achieved improved classification accuracy and demonstrated the effectiveness of neural networks for underwater sonar signal classification.

# References
1. Gorman, R. P., & Sejnowski, T. J. (1988). *Analysis of Hidden Units in a Layered Network Trained to Classify Sonar Targets. Neural Networks.*
2. TensorFlow Documentation
3. Keras Documentation
4. Scikit-Learn Documentation